In [2]:
import time
import numpy
import json
import base64

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from concrete.ml.common.utils import CiphertextFormat
import json
import concrete_ml_extensions as fhext
from concrete.ml.sklearn import DecisionTreeClassifier as ConcreteDecisionTreeClassifier
from concrete.compiler import check_gpu_available
import json
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from concrete.ml.sklearn import DecisionTreeClassifier as ConcreteDecisionTreeClassifier
import pickle as pkl

# Lets create a synthetic data-set
x, y = make_classification(n_samples=100, class_sep=2, n_features=30, random_state=42)

# Split the data-set into a train and test set
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Now we train in the clear and quantize the weights
model = ConcreteDecisionTreeClassifier(n_bits=8)
model.fit(X_train, y_train)

# We can simulate the predictions in the clear
y_pred_clear = model.predict(X_test)

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


In [3]:
model.compile(X_train[:10, :], ciphertext_format=CiphertextFormat.TFHE_RS)

In [4]:
# crypto_params = json.loads(fhext.get_crypto_params_radix())

sk, tfhers_pk, pk, lwe_sk = model.keygen_tfhers()

with open("lweServerKey.bin", "wb") as f:
    f.write(lwe_sk)

with open("serverKey.bin", "wb") as f:
    f.write(pk)

In [5]:
q_input = model.quantize_input(X_test[[0]])

q_input_enc = model.encrypt_tfhers(q_input, tfhers_sk=sk)

In [6]:
q_y_enc = model.run_tfhers(q_input_enc, tfhers_pk=tfhers_pk)

with open("encrypted_input.bin", "wb") as f:
    f.write(q_y_enc)

tfhers<uint8, 3, 2, params=crypto_params<lwe_dim=902, glwe_dim=1, poly_size=4096, pbs_base_log=15, pbs_level=2, lwe_noise_distribution=1.0994794733558207e-06, glwe_noise_distribution=2.168404344971009e-19, encryption_key_choice=BIG>> ici
tfhers_int_description<width=8, signed=0, msg_mod=4, carry_mod=8, degree=3, lwe_size=4097, n_cts=4, noise_level=18446744073709551615, ks_first=1> la


In [7]:
q_y = model.decrypt_tfhers(q_y_enc, tfhers_sk=sk)

In [8]:
y0 = model.post_processing(model.dequantize_output(q_y))

In [13]:
assert y0.argmax() == y_test[0]